In [0]:
%sql
CREATE OR REPLACE VIEW retail_q.retail_semantic.retail_metrics
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: retail_q.retail_gold.fact_sales
  comment: Centralized retail metrics for sales analysis across products and time dimensions
  
  joins:
    - name: product
      source: retail_q.retail_gold.dim_product
      on: source.product_id = product.product_id
    
    - name: calendar
      source: retail_q.retail_gold.dim_calender
      on: source.transaction_date = calendar.date
  
  dimensions:
    # Transaction Date Dimensions
    - name: Transaction Date
      expr: transaction_date
      display_name: Transaction Date
      comment: Date of the retail transaction
      format:
        type: date
        date_format: year_month_day
      synonyms:
        - sale date
        - date
    
    - name: Transaction Month
      expr: DATE_TRUNC('MONTH', transaction_date)
      display_name: Transaction Month
      comment: Month when the transaction occurred
      format:
        type: date
        date_format: locale_short_month
      synonyms:
        - month
        - sales month
    
    - name: Transaction Quarter
      expr: DATE_TRUNC('QUARTER', transaction_date)
      display_name: Transaction Quarter
      comment: Quarter when the transaction occurred
      format:
        type: date
        date_format: year_month_day
      synonyms:
        - quarter
    
    # Transaction Attributes
    - name: Payment Mode
      expr: payment_mode
      display_name: Payment Mode
      comment: Method of payment used in the transaction
      synonyms:
        - payment method
        - payment type
    
    - name: Sales Channel
      expr: sales_channel
      display_name: Sales Channel
      comment: Channel through which the sale was made
      synonyms:
        - channel
        - distribution channel
    
    - name: Stage Name
      expr: stage_name
      display_name: Opportunity Stage
      comment: Sales stage of the opportunity
      synonyms:
        - sales stage
        - opportunity status
    
    - name: Store ID
      expr: store_id
      display_name: Store ID
      comment: Identifier of the store where transaction occurred
      synonyms:
        - store
    
    - name: Opportunity Name
      expr: opportunity_name
      display_name: Opportunity Name
      comment: Name of the sales opportunity
      synonyms:
        - opportunity
    
    # Product Dimensions
    - name: Product ID
      expr: product.product_id
      display_name: Product ID
      comment: Unique identifier for the product
    
    - name: Product Name
      expr: product.product_name
      display_name: Product Name
      comment: Name of the product sold
      synonyms:
        - product
    
    - name: Category
      expr: product.category
      display_name: Product Category
      comment: High-level product category
      synonyms:
        - product category
    
    - name: Subcategory
      expr: product.subcategory
      display_name: Product Subcategory
      comment: Detailed product subcategory
      synonyms:
        - product subcategory
        - sub category
    
    - name: Brand
      expr: product.brand
      display_name: Brand
      comment: Product brand name
      synonyms:
        - product brand
        - manufacturer
    
    - name: Supplier Name
      expr: product.supplier_name
      display_name: Supplier Name
      comment: Name of the product supplier
      synonyms:
        - supplier
        - vendor
    
    # Calendar Dimensions
    - name: Year
      expr: calendar.year
      display_name: Year
      comment: Year of the transaction
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
    
    - name: Month
      expr: calendar.month
      display_name: Month Number
      comment: Month number (1-12)
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
    
    - name: Quarter
      expr: calendar.quarter
      display_name: Quarter Number
      comment: Quarter of the year (1-4)
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
    
    - name: Week
      expr: calendar.week
      display_name: Week Number
      comment: Week number of the year
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - week of year
    
    - name: Day of Week
      expr: calendar.day_of_week
      display_name: Day of Week
      comment: Day of the week (1=Sunday, 7=Saturday)
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - weekday
    
    - name: Day of Month
      expr: calendar.day_of_month
      display_name: Day of Month
      comment: Day of the month (1-31)
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
  
  measures:
    # Core Count Measures
    - name: Transaction Count
      expr: COUNT(DISTINCT transaction_id)
      display_name: Transaction Count
      comment: Total number of unique transactions
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - number of transactions
        - transaction volume
        - sales count
    
    - name: Product Count
      expr: COUNT(DISTINCT product_id)
      display_name: Product Count
      comment: Number of unique products sold
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - number of products
        - unique products
    
    # Revenue Measures
    - name: Total Revenue
      expr: SUM(selling_price * quantity)
      display_name: Total Revenue
      comment: Total revenue from all sales before discounts
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - revenue
        - gross sales
        - total sales
    
    - name: Total Discount
      expr: SUM(discount_amount)
      display_name: Total Discount
      comment: Total discount amount given to customers
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - discount
        - total discounts
    
    - name: Net Revenue
      expr: SUM(selling_price * quantity - discount_amount)
      display_name: Net Revenue
      comment: Revenue after discounts (net sales)
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - net sales
        - revenue after discount
    
    - name: Opportunity Amount
      expr: SUM(amount)
      display_name: Opportunity Amount
      comment: Total opportunity amount from CRM
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - opportunity value
    
    # Quantity Measures
    - name: Total Quantity
      expr: SUM(quantity)
      display_name: Total Quantity Sold
      comment: Total quantity of products sold (units)
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - quantity sold
        - units sold
        - volume
    
    # Average Measures (using MEASURE composition)
    - name: Average Transaction Value
      expr: MEASURE(`Total Revenue`) / MEASURE(`Transaction Count`)
      display_name: Average Transaction Value
      comment: Average revenue per transaction
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - avg transaction value
        - average order value
        - AOV
    
    - name: Average Selling Price
      expr: SUM(selling_price * quantity) / SUM(quantity)
      display_name: Average Selling Price
      comment: Average price per unit sold
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - avg price
        - average price per unit
        - unit price
    
    - name: Average Discount
      expr: MEASURE(`Total Discount`) / MEASURE(`Transaction Count`)
      display_name: Average Discount per Transaction
      comment: Average discount amount per transaction
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - avg discount
    
    - name: Average Items per Transaction
      expr: MEASURE(`Total Quantity`) / MEASURE(`Transaction Count`)
      display_name: Average Items per Transaction
      comment: Average number of items sold per transaction
      format:
        type: number
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - avg basket size
        - items per order
    
    # Percentage Measures
    - name: Discount Percentage
      expr: SUM(discount_amount) / SUM(selling_price * quantity) * 100
      display_name: Discount Percentage
      comment: Discount as percentage of total revenue
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - discount rate
        - discount ratio
$$

In [0]:
%skip
%sql
-- Test the metric view with a sample query
SELECT 
  `Brand`,
  `Category`,
  MEASURE(`Total Revenue`) AS total_revenue,
  MEASURE(`Transaction Count`) AS transaction_count,
  MEASURE(`Average Transaction Value`) AS avg_transaction_value
FROM retail_q.retail_semantic.retail_metrics
GROUP BY ALL
ORDER BY total_revenue DESC
LIMIT 10